# **텍스트 요약**

**내장 체인(built-in chains)** 과 [LangGraph](https://langchain-ai.github.io/langgraph/)를 사용하여 **텍스트 요약**을 시연합니다.  

PDF, Notion 페이지, 고객 질문 등 **여러 문서**를 가지고 있고 그 내용을 **요약** 하고 싶다고 가정해 봅시다.

**대규모 언어 모델(LLM)** 은 텍스트를 이해하고 요약하는 데 뛰어난 능력을 가지고 있어 이 작업에 매우 적합합니다.

**검색 증강 생성(Retrieval-Augmented Generation, RAG)** 측면에서 볼 때, 텍스트를 요약하는 것은 많은 수의 검색된 문서에서 **핵심 정보를 추출**하여 LLM에 컨텍스트를 제공하는 데 도움을 줄 수 있습니다.

 **여러 문서의 내용을 LLM을 사용하여 요약하는 방법**을 단계별로 살펴보겠습니다.

![Image description](https://github.com/langchain-ai/langchain/blob/master/docs/static/img/summarization_use_case_1.png?raw=1)

여기에서 다룰 개념은 다음과 같습니다:

- **문서 로더**의 사용, 특히 [**WebBaseLoader**](https://python.langchain.com/api_reference/community/document_loaders/langchain_community.document_loaders.web_base.WebBaseLoader.html)를 사용해 HTML 웹페이지의 콘텐츠를 로드하는 방법.

- **문서를 요약하거나 결합하는 두 가지 방법:**
   1. **Stuff**: 문서를 단순히 프롬프트에 연결합니다.  
   2. **Map-Reduce**: 더 많은 문서 집합에 사용됩니다. 문서를 배치로 나눈 후 각 배치를 요약하고, 마지막으로 요약된 결과를 다시 요약합니다.  


**summarizer**를 구축할 때 중심이 되는 질문은 **어떻게 문서를 LLM의 컨텍스트 윈도우에 전달할 것인가?** 입니다. 이 문제를 해결하기 위한 두 가지 일반적인 접근 방식이 있습니다:

1. **`Stuff`:**  
   - 모든 문서를 **하나의 프롬프트에 "채워 넣는"** 가장 간단한 방법입니다.  
   - 이 방법은 단순하지만 대규모 문서에는 적합하지 않을 수 있습니다.  
<p></p>

2. **`Map-reduce`:**  
   - 먼저 각 문서를 개별적으로 요약하는 **"Map" 단계**를 거칩니다.  
   - 이후, 요약된 결과를 결합하여 최종 요약을 만드는 **"Reduce" 단계**를  수행합니다.  
<p></p>

**`Map-reduce`** 는 특히 **부분 문서(sub-document)** 의 이해가 **이전 맥락에 의존하지 않는 경우** 효과적입니다. 예를 들어, **여러 개의 짧은 문서 모음** 을 요약할 때 유용합니다.  

반면, 소설이나 연속된 흐름이 중요한 긴 텍스트를 요약할 때는 **반복적 개선(iterative refinement)** 접근 방식이 더 효과적일 수 있습니다.

![Image description](https://github.com/langchain-ai/langchain/blob/master/docs/static/img/summarization_use_case_2.png?raw=1)

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano", model_provider="openai")
# model = init_chat_model("gemini-2.5-flash", model_provider="google_genai")

먼저 문서를 불러옵니다. 블로그 게시글을 불러오기 위해 [**WebBaseLoader**](https://python.langchain.com/api_reference/community/document_loaders/langchain_community.document_loaders.web_base.WebBaseLoader.html)를 사용할 것입니다.

In [3]:
from langchain_community.document_loaders import WebBaseLoader

#  WebBaseLoader 객체 생성
loader = WebBaseLoader("https://botpress.com/ko/blog/llm-agents")

# load(): 지정된 URL의 내용을 가져와 LangChain 문서 형식으로 변환
docs = loader.load()

len(docs[0].page_content)

USER_AGENT environment variable not set, consider setting it to identify your requests.


10481

###  **Stuff: 단일 LLM 호출로 요약하기**

[**`create_stuff_documents_chain`**](https://python.langchain.com/api_reference/langchain/chains/langchain.chains.combine_documents.stuff.create_stuff_documents_chain.html)을 사용할 수 있습니다. 특히  **큰 컨텍스트 윈도우(Context Window)** 를 지원하는 모델을 사용할 때 유용합니다:

- 128k 토큰 OpenAI `gpt-5` 
- 200k 토큰 Anthropic `claude-3-5-sonnet-20240620`

이 체인은 문서 목록을 가져와 **모든 문서를 하나의 프롬프트로 삽입**한 뒤, 그 프롬프트를 **LLM에 전달**합니다.

In [4]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

#  프롬프트 정의
prompt = ChatPromptTemplate(
    [("system", "다음을 간결하게 요약하세요, 한국어로 요약하세요.\\n\\n{context}")]
)

# 체인 인스턴스 생성
chain = create_stuff_documents_chain(model, prompt)
     
# 체인 실행 (invoke)
result = chain.invoke({"context": docs})
print(result)

다음 내용을 간결하게 요약합니다.

- 플랫폼 개요
  - Botpress: LLM 기반 자율 에이전트 플랫폼. 상담원 스튜디오, 자율 엔진, 지식 베이스, 휴먼 핸드오프, 다채널(WhatsApp, Slack 등)과 통합 포함. 다양한 LLM 공급자(OpenAI, Anthropic 등) 지원. 엔터프라이즈 및 에이전시용 솔루션 제공.

- 에이전트의 작동 원리
  - 검색(RAG), 추론 프롬프트, 메모리(단기/장기), 도구 사용(외부 시스템 연동) 네 가지 기능을 결합하여 자율적 다단계 워크플로우 수행.
  - 예: CRM 데이터 조회 후 분석, 이메일/캘린더 연동으로 일정 관리 등.

- 핵심 기능 4가지
  - 언어 모델: 두뇌 역할로 응답의 품질 결정.
  - 메모리: 대화 맥락/선호 저장으로 연속성 향상.
  - 도구 사용: API/데이터베이스 연동으로 실무 작업 수행.
  - 계획: 작업을 단계로 나누고 피드백 여부에 따라 계획을 조정.

- 에이전트의 4가지 유형
  - 대화형: 고객 지원, 리드 생성 등.
  - 작업 지향: 특정 작업 달성에 집중.
  - 크리에이티브: 콘텐츠 제작 도구.
  - 협업: 인간과 협력하는 엔터프라이즈 AI 에이전트.

- 주요 사용 사례
  - 고객 지원 자동화, 영업/리드 제너레이션, 내부 HR/IT 지원 등. 다양한 산업에 걸쳐 텍스트 처리와 워크플로 자동화에 활용.

- 구축/배포 방법(6단계)
  - 1) 목표 정의, 2) AI 플랫폼 선택, 3) 구성(모델 선택/미세조정), 4) 도구 통합, 5) 테스트 및 개선, 6) 배포 및 모니터링.

- 성능 개선 및 트레이드오프(4가지)
  - RAG, 미세 조정, N샷 프롬프트, 프롬프트 체이닝(신속한 엔지니어링).
  - 각 방법은 품질 향상과 함께 비용/시간/지연의 trade-off가 있음.

- 보안/리소스
  - SOC 2, GDPR 준수. Academy, 문서, 블로그, 커뮤니티 등 다양한 학습/지원 자원 제공.

- 자주 묻는 질문 요약
  - LLM

###  **스트리밍**

결과를 **토큰 단위로 스트리밍**할 수도 있습니다.

In [5]:
# 체인 실행 및 스트리밍
for token in chain.stream(
    {"context": docs}
):
    print(token, end="|")

|다|음|은| “|LL|M| 에|이|전|트| 완|전| 가|이드|(|202|5|)”|의| 핵|심|을| 간|결|하게| 정|리|한| 요|약|입니다|.

|-| 무엇|인가|요|
| | -| L|LM| 에|이|전|트|는| 큰| 언|어| 모델|을| 이용|해| 언|어| 이해|·|대|화|·|작|업| 수행|을| 자|율|적으로| 수행|하는| AI| 도|구|.| 검색|·|추|론|·|메|모|리|·|도|구| 사용| 등을| 결|합|해| 다|단|계| 워|크|플|로|우|를| 처리|합니다|.

|-| 핵|심| 구성|요|소|
| | -| 검색|(R|AG|):| 실|시간| 데이터|에| 접근|
| | -| 추|론|:| 의|사|결|정| 방법|에| 대한| 고|급| 프|롬|프트| 활용|
| | -| 메|모|리|:| 단|기| 대|화| 기록|과| 장|기| 지|식| 저장|으로| 맥|락| 유지|
| | -| 도|구| 사용|:| 외|부| 시스템|(API|/|DB| 등|)|과| 연|동|해| 데이터| 조회| 및| 작업| 수행|
| | -| 계획|:| 복|잡|한| 작업|을| 단계|로| 쪼|개|는| 전략|(|피|드|백| 여부|에| 따라| 계획| 방식|이| 달|라|짐|)

|-| L|LM| 에|이|전|트|의| |4|대| 핵|심| 기능|
| | -| 언|어| 모델|:| 두|뇌| 역할|,| 응|답|의| 질|·|적|합|성| 결정|
| | -| 메|모|리|:| 과|거| 상|호|작|용| 기억|으로| 맥|락| 강화|
| | -| 도|구| 사용|:| 외|부| 시스템|과|의| 상|호|작|용|으로| 실|시간| 작업| 수행|
| | -| 계획|:| 작업| 흐|름|을| 체|계|적으로| 설|계|하고| 필요|시| 피|드|백| 반|영|

|-| 에|이|전|트|의| |4|가지| 유형|
| | -| 대|화|형| 상담|원|:| 고객|지원| 등| 대|화| 중심|
| | -| 작업| 지|향| 에|이|전|트|:| 특정| 작업|/|목|표| 달|성|에| 집중|
| | -| 크|리에|이|티|브| 에|이|전|트|:| 콘텐츠| 제작| 등| 창|의|적| 작업|
|

--------------------
### **Map-Reduce: 병렬 처리를 통한 긴 텍스트 요약**

**Map-Reduce 접근법**에서는 먼저 각 문서를 **LLM을 사용하여 개별 요약(map 단계)** 한 후, 이러한 요약들을 **하나의 글로벌 요약으로 통합(reduce 단계)** 합니다.

**Map 단계**는 일반적으로 입력 문서에 대해 **병렬 처리**됩니다.

**LangGraph**는 `langchain-core`를 기반으로 구축되었으며, [**Map-Reduce**](https://langchain-ai.github.io/langgraph/how-tos/map-reduce/) 워크플로우를 지원하므로 이 문제를 해결하기에 매우 적합합니다:

- **LangGraph**는 각 단계(예: 연속적인 요약)가 **스트리밍될 수 있도록 지원**하여 실행을 더 세밀하게 제어할 수 있습니다.  
- **LangGraph**의 [**체크포인트(checkpointing)**](https://langchain-ai.github.io/langgraph/how-tos/persistence/) 기능은 오류 복구, 인간 검토(human-in-the-loop) 워크플로우 확장, 대화형 애플리케이션 통합을 더욱 쉽게 지원합니다.  
- **LangGraph**는 간단하고 유연하게 수정 및 확장할 수 있습니다. 아래에서 자세히 확인해보겠습니다.



### **Map/Reduce Template 작성.**

먼저 **Map 단계**에 사용할 **프롬프트**를 정의해 보겠습니다. 이 단계에서는 `stuff` 접근법과 동일한 요약 프롬프트를 사용할 수 있습니다.

In [6]:
from langchain_core.prompts import ChatPromptTemplate

# Map 단계에 사용할 프롬프트 정의
map_prompt = ChatPromptTemplate.from_messages(
    [("system", "다음을 간결하게 요약하세요. 한국어로 요약하세요.\\n\\n{context}")]
)

문서 매핑 결과를 받아 **하나의 출력으로 통합하는 Reduce 프롬프트**를 정의합니다.

In [7]:
# Reduce 단계에서 사용할 프롬프트 정의
reduce_template = """
다음은 요약된 내용들의 모음입니다:
{docs}
이것들을 분석하여 주요 주제를 담은 최종 통합 요약으로 압축하세요.
"""

reduce_prompt = ChatPromptTemplate.from_messages(
    [("human", reduce_template)]
)

### **LangGraph를 통한 오케스트레이션**

**Map-Reduce 흐름**은 LLM의 **컨텍스트 윈도우(context window)** 보다 텍스트가 긴 경우 특히 유용합니다. 긴 텍스트의 경우, Reduce 단계에서 요약해야 할 컨텍스트가 **모델의 컨텍스트 윈도우 크기를 초과하지 않도록** 관리하는 메커니즘이 필요합니다.  

여기서는 요약을 **재귀적으로 "압축(collapsing)"** 하는 방식을 구현합니다:  
- 입력 텍스트는 **토큰 제한(token limit)** 에 따라 분할됩니다.  
- 각 부분(partition)에 대한 요약이 생성됩니다.  
- 이 단계는 요약된 결과의 총 길이가 원하는 한도 내에 도달할 때까지 반복됩니다.  
- 이를 통해 **임의 길이의 텍스트를 효과적으로 요약**할 수 있습니다.

먼저 블로그 게시글을 더 작은 **"하위 문서(sub documents)"** 로 분할합니다.

In [8]:
from langchain_text_splitters import CharacterTextSplitter

# OpenAI의 tiktoken 기반 인코딩을 적용
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000,   
    chunk_overlap=0   
)

# 문서를 하위 문서로 분할
split_docs = text_splitter.split_documents(docs)

# 생성된 하위 문서의 개수 출력
len(split_docs)

Created a chunk of size 2267, which is longer than the specified 1000
Created a chunk of size 6879, which is longer than the specified 1000
Created a chunk of size 4369, which is longer than the specified 1000
Created a chunk of size 1748, which is longer than the specified 1000
Created a chunk of size 2090, which is longer than the specified 1000


13

------------
다음으로, **그래프(graph)** 를 정의합니다.  

여기서는 **"압축(collapsing)" 단계**를 설명하기 위해 인위적으로 **1,000 토큰**이라는 낮은 최대 토큰 길이를 설정합니다.

아래 코드에서 async 와 await 는 **비동기적으로 LLM을 호출** 하여 응답을 기다리고, 여러 문서를 **병렬**로 처리하는 역할을 합니다.

In [14]:
from langchain_core.documents import Document 
from langgraph.graph import END, START, StateGraph, MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

import operator
from typing import Annotated, List, Literal, TypedDict

# 최대 토큰 길이 설정
token_max = 1000

# 문서 길이 계산 함수
def length_function(documents: List[Document]) -> int:
    # 주어진 문서 목록의 총 토큰 수 반환
    return sum(model.get_num_tokens(doc.page_content) for doc in documents)

# State 정의
class SummarizationState(TypedDict):
    documents: List[Document]           # 원본 문서 목록
    summaries: Annotated[List[str], operator.add]  # 개별 요약 목록
    final_summary: str                 # 최종 요약 결과

async def summarize_document(content: str) -> str:
    """개별 문서를 요약하는 함수"""
    prompt = map_prompt.invoke({"context": content})
    response = await model.ainvoke(prompt)
    return response.content

async def map_summaries(state: SummarizationState):
    """Map 단계: 모든 문서를 병렬로 요약"""
    tasks = [summarize_document(doc.page_content) for doc in state["documents"]]
    summaries = await asyncio.gather(*tasks)
    return {"summaries": summaries}

async def reduce_summaries(state: SummarizationState):
    """Reduce 단계: 요약들을 통합"""
    if len(state["summaries"]) == 1:
        return {"final_summary": state["summaries"][0]}
    
    # 요약들을 Document로 변환
    summary_docs = [Document(page_content=summary) for summary in state["summaries"]]
    
    # 토큰 수가 제한을 초과하면 재귀적으로 압축
    if length_function(summary_docs) > token_max:
        # 절반씩 나누어 처리
        mid = len(summary_docs) // 2
        first_half = summary_docs[:mid]
        second_half = summary_docs[mid:]
        
        # 각 절반을 개별적으로 요약
        first_summary = await _reduce_summaries(first_half)
        second_summary = await _reduce_summaries(second_half)
        
        # 두 요약을 다시 통합
        combined_docs = [Document(page_content=first_summary), Document(page_content=second_summary)]
        final_summary = await _reduce_summaries(combined_docs)
    else:
        final_summary = await _reduce_summaries(summary_docs)
    
    return {"final_summary": final_summary}

async def _reduce_summaries(docs: List[Document]) -> str:
    """문서 목록을 하나의 요약으로 통합"""
    combined_content = "\n\n".join([doc.page_content for doc in docs])
    prompt = reduce_prompt.invoke({"docs": combined_content})
    response = await model.ainvoke(prompt)
    return response.content

# 그래프 구성
graph = StateGraph(SummarizationState)
graph.add_node("map_summaries", map_summaries)
graph.add_node("reduce_summaries", reduce_summaries)

# Graph Flow: START -> map_summaries -> reduce_summaries -> END
graph.add_edge(START, "map_summaries")
graph.add_edge("map_summaries", "reduce_summaries")
graph.add_edge("reduce_summaries", END)

# 메모리 체크포인터 추가 (선택사항)
memory = MemorySaver()
app = graph.compile(checkpointer=memory)
app;

애플리케이션을 실행할 때 **그래프의 단계 순서**를 관찰하기 위해 스트리밍할 수 있습니다. 아래에서는 간단히 각 단계의 이름을 출력합니다.  

그래프에 **반복(loop)** 이 포함되어 있기 때문에, 실행 시 [**recursion_limit**](https://langchain-ai.github.io/langgraph/reference/errors/#langgraph.errors.GraphRecursionError)을 지정하는 것이 유용할 수 있습니다.  

지정된 한도를 초과하면 **GraphRecursionError**가 발생합니다.

In [10]:
async def run_summarization():
    """
    요약 실행 함수
    - app.ainvoke()를 호출하여 LangGraph 워크플로우 실행
    - recursion_limit: 최대 재귀 호출 횟수
    - configurable.thread_id: 실행 세션을 구분하기 위한 식별자
    """
    config = {
        "recursion_limit": 10,  # 최대 재귀 단계 (collapse 반복 횟수 등)
        "configurable": {"thread_id": "summarization_session"}  # 세션 식별자
    }
    result = await app.ainvoke(
        {"documents": split_docs},  # 입력으로 분할된 문서 전달
        config=config
    )
    return result  # 그래프 실행 결과 반환

# 실행 및 결과 출력
async def main():
    
    result = await run_summarization()
    
    print("최종 요약:")
    print(result["final_summary"])

# 메인 함수 실행 (비동기)
await main()

최종 요약:
다음 내용을 하나의 최종 통합 요약으로 압축합니다.

주요 요지
- LLM 에이전트는 언어 이해, 기억, 도구 사용, 계획의 4대 기능을 바탕으로 자율적이고 다단계 워크플로우를 수행합니다. 이는 다양한 부서에서 워크플로 자동화와 대화형 서비스를 구현하는 강력한 도구로 자리 잡고 있습니다.
- 구현의 핵심은 목표 정의와 안전성/거버넌스의 사전 수립, 도구 연동 및 시스템 통합, 지속적 모니터링에 있습니다. 기술 전략으로는 RAG(실시간 정보 활용), 프롬프트 엔지니어링/미세조정, N샷 프롬프트, 체인 오브 Thought가 균형 있게 사용됩니다.
- 운영 측면에서 6단계 빌드 로드맷(목표 정의 → 플랫폼 선택 → 에이전트 아키텍처 결정 → 의사결정 루프 설계 → 학습/데이터 전략 → 안전성/윤리 평가 및 벤치마크 → 배포/운영)과 7대 축의 체계적 구축이 권장됩니다. API/데이터베이스/CRM 등 외부 자원과의 연동이 핵심적이며, 모니터링과 피드백 루프를 통한 지속적 개선이 필수입니다.

핵심 구성 요소
- 4대 핵심 기능: 언어 모델(두뇌), 메모리(단기/장기 기억으로 맥락 유지), 도구 사용(외부 시스템 연동), 계획(단계적 작업 분해 및 피드백에 따른 개선).
- 4가지 에이전트 유형: 대화형 상담원, 작업 지향 에이전트, 크리에이티브 에이전트, 협업 에이전트.
- 기술 요소: RAG를 통한 최신 정보 접근, 프롬프트 엔지니어링/미세조정으로 품질 향상, 메모리 구조를 통한 의사결정의 적응성과 개인화.

실무적 시사점/운영
- 활용 분야: 마케팅, 데이터 분석, 규정 준수, 법률/의료 지원, 재무, 교육 등 다방면에서 활용 가능.
- 대표 사례: 24/7 고객 지원 및 에스컬레이션, 영업 리드 생성과 후속 자동화, 내부 HR/IT 지원으로 비용·시간 절감.
- 도구/생태계: API, 데이터베이스, CRM 등과의 연동이 핵심. Botpress 등 엔터프라이즈용 플랫폼 사례를 통해 관리성과 보안 강화를 강조.
- 한계와 주의점: 환각, 개인정보 보호 

-----------------------------
#### astream : 실행 중간 단계별 이벤트를 스트리밍으로 받아서 진행 상황/부분 결과를 즉시 확인.

In [11]:
# 스트리밍 방식으로 실행 
async def stream_summarization():
    """스트리밍 방식으로 요약 실행"""
    config = {
        "recursion_limit": 10,
        "configurable": {"thread_id": "summarization_session"}
    }
    print("요약 프로세스 시작...")
    async for step in app.astream(
        {"documents": split_docs},
        config=config
    ):
        for node_name, node_output in step.items():
            print(f"단계: {node_name}")
            
            # 각 단계별 출력 확인
            if node_name == "map_summaries":
                print(f"  - {len(node_output.get('summaries', []))}개 문서 요약 완료")
                # 첫 번째 요약 미리보기
                if node_output.get('summaries'):
                    preview = node_output['summaries'][0][:100] + "..." if len(node_output['summaries'][0]) > 100 else node_output['summaries'][0]
                    print(f"  - 첫 번째 요약 미리보기: {preview}")
            
            elif node_name == "reduce_summaries":
                if "final_summary" in node_output:
                    print(f"  - 최종 요약 완료!")
                    print(f"  - 최종 요약: {node_output['final_summary']}")
                else:
                    print(f"  - 요약 통합 중...")

# 스트리밍 실행
await stream_summarization()

요약 프로세스 시작...
단계: map_summaries
  - 13개 문서 요약 완료
  - 첫 번째 요약 미리보기: - 요약: LLM 에이전트 완전 가이드는 플랫폼 구성 요소, 채널/통합, 공급자, 솔루션 유형 및 산업·부서별 사용 사례를 포괄한다.

- 플랫폼 특징
  - 상담원 스튜디오: 신...
단계: reduce_summaries
  - 최종 요약 완료!
  - 최종 요약: 다음은 제시된 내용을 한 눈에 담을 수 있도록 압축한 최종 통합 요약입니다.

한 줄 요약
- LLM 에이전트는 언어 이해, 기억(메모리), 도구 사용, 계획의 4대 기능으로 자율적으로 다단계 워크플로우를 수행하고, 다양한 시스템·채널과의 통합을 통해 기업의 고객지원부터 내부 운영까지 폭넓은 자동화를 가능하게 하는 보안 중심의 플랫폼이다.

핵심 주제 요약
- 정의와 핵심 역량
  - 정의: 의도를 이해하고 도구를 사용해 작업을 수행하는 자율형 AI 도구.
  - 핵심 역량: 언어 이해/생성, 메모리 관리, 도구 활용, 계획 인지로 복잡한 작업까지 자동화.
  - 개선 기술: RAG(실시간 정보 검색), 미세 조정, N샷 프롬프트, 프롬프트 체이닝으로 품질과 안정성 향상.

- 구축·운영의 구성 요소
  - 플랫폼 구성: 상담원 스튜디오(빠른 구축/커스터마이즈), 자율 엔진 LLMs, 지식 베이스, 휴먼 핸드오프.
  - 데이터/채널/통합: 표 데이터 관리, WhatsApp/Instagram/Messenger/Slack 등 채널 지원, HubSpot/Notion/Jira/Calendly 등 외부 시스템 연동.
  - 공급자와 포지션: OpenAI, Anthropic, Groq, Hugging Face 등 다양한 LLM 공급자 활용 가능; 엔터프라이즈/에이전시/개발자(API 중심)로 구분해 활용.

- 활용 사례와 산업
  - 산업 적용 범위: 전자상거래, 교육, 금융, 호스피탈리티 등 모든 산업.
  - 부서별/사용 사례: 판매, 엔지니어링, ITSM 포함 다양한 부서의 워크플로